# NRGS in other LMs — combined figures (App. "Checking if NRGS Holds in Other LMs")

One figure per experiment, one row per LM, same colour scheme as the Coder-32B figures:
- `tvp_grids_by_lm` — task span vs code-progress span: residual transfer (Fig 27 right) + KV on 250 goal heads, V×1/2/4 (Fig 28 (i)).
- `kv_barcharts_by_lm` — pivot-onwards KV transfer of the task span (Fig 22).
- `q_barcharts_by_lm` — query transfer with gaze-select-restrict (Fig 26).

Other LMs' NRGS readouts use the replace strategy (α 1, layers 35-45; arm suffix `__ros-replace-a1-L35-46`).
Outputs → `notebooks/Figures/other_lms/`.

In [ ]:
import os, sys
_d=os.getcwd()
while _d!='/' and not os.path.isdir(os.path.join(_d,'src')): _d=os.path.dirname(_d)
sys.path.insert(0,_d); os.chdir(_d)
from src.utils import env_utils
sys.path.insert(0, os.path.join(env_utils.PROJECT_ROOT,'notebooks','fig_n_table_generators'))

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import figstyle as F; F.set_style()

MODELS = ['Qwen3-32B', 'Olmo-3.1-32B-Instruct']
GROUP = 'other_lms'
CODER = 'Qwen2.5-Coder-32B-Instruct'
COLOR = F.MODEL_COLOR[CODER]          # same colour for every LM (matches the Coder-32B figures)
ROS = lambda m: '' if m == CODER else '__ros-replace-a1-L35-46'
DIFF = lambda m: os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'diff', m)

DTS   = ['int', 'str', 'list[int]', 'list[str]']
LANGS = ['python-python', 'python-cpp', 'cpp-python', 'cpp-cpp']
LANG_LBL = {'python-python': 'py$\\to$py', 'python-cpp': 'py$\\to$cpp', 'cpp-python': 'cpp$\\to$py', 'cpp-cpp': 'cpp$\\to$cpp'}
KV_DT = {'int': 'int', 'str': 'str', 'list[int]': 'list_int', 'list[str]': 'list_str'}
Q_DT  = {'int': 'int', 'str': 'str', 'list[int]': 'list(int)', 'list[str]': 'list(str)'}
VAR_LBL = {'order_matched': 'order-matched', 'order_diff': 'order-diff'}

def _read(p):
    d = json.load(open(p)); return d[0] if isinstance(d, list) else d

def kv_grid(m):
    arm = os.path.join(DIFF(m), 'path_patch', 'I1-off__I2-off__I3-on-V4__win-span__span-list+gap__cppnote-sys__a1__th-goal_250__gh-goal_250__ro-goal_250' + ROS(m))
    return {dt: {lg: (lambda e: (e['causal_trigger_rate'], e['readout_shift_rate']))(_read(os.path.join(arm, f'{KV_DT[dt]}__{lg}', 'meta.json')))
                 for lg in LANGS} for dt in DTS}

def q_grid(m, v):
    arm = os.path.join(DIFF(m), 'diff_trigger', f'ph-goal_250__a2__piv-eq__cl-ppgazerestrict-anceq{ROS(m)}__seq')
    return {dt: {lg: (lambda e: (e['causal_trigger_rate'], e['readout_shift_rate']))(_read(os.path.join(arm, v, f'{Q_DT[dt]}__{lg}', 'summary.json')))
                 for lg in LANGS} for dt in DTS}

def mean_over(g, axis):
    cats = LANGS if axis == 'lang' else DTS
    cell = (lambda c: [g[dt][c] for dt in DTS]) if axis == 'lang' else (lambda c: [g[c][lg] for lg in LANGS])
    return cats, np.array([np.mean([x[0] for x in cell(c)]) for c in cats]), np.array([np.mean([x[1] for x in cell(c)]) for c in cats])

def grouped_metric_bars(ax, cats, labels, causal, readout, w=0.38):
    x = np.arange(len(cats))
    ax.bar(x - w/2, causal,  w, **F.metric_kw('causal',  COLOR))
    ax.bar(x + w/2, readout, w, **F.metric_kw('readout', COLOR))
    for cont in ax.containers: ax.bar_label(cont, fmt='%.2f', padding=1.5, fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.08); ax.set_ylabel('shift rate')

def legend_top(fig, y):
    hs = [Patch(label='causal-shift',  facecolor=COLOR, edgecolor='white', linewidth=0.6),
          Patch(label='readout-shift', facecolor=COLOR, edgecolor='white', linewidth=0.6, hatch=F.HATCH)]
    fig.legend(handles=hs, labels=[h.get_label() for h in hs], loc='lower center', bbox_to_anchor=(0.5, y),
               ncol=2, frameon=False, handlelength=1.6, columnspacing=1.4)

In [ ]:
# ── task span vs code-progress span: one row per LM (residual | KV on 250 goal heads, V×1, 2, 4) ──
def tvp_summary(m):
    root = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'notebook_caches', 'task_vs_progress', m)
    return {d: json.load(open(os.path.join(root, f'{s}__python-python__seed1123__mx64__ro0__task+gap.summary.json'))) for d, s in KV_DT.items()}

def tvp_grid(T, method, heads=None, v=1.0):
    name = lambda span: 'clean DST (no-op)' if method == 'none' else f"{method}{'·' + heads if heads else ''} · {span} · V×{v:g}"
    r = lambda n: float(np.mean([T[d][n]['causal_shift'] for d in DTS]))
    return np.array([[r('clean DST (no-op)'), r(name('progress'))], [r(name('task')), r(name('task+progress'))]])

CMAP_KV  = LinearSegmentedColormap.from_list('tvp', ['#f3f7fb', COLOR])
CMAP_RES = LinearSegmentedColormap.from_list('tvp_res', ['#fbf3f2', 'firebrick'])
def draw(ax, M, title, cmap, ylab):
    ax.imshow(M, cmap=cmap, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=10, color='white' if M[i, j] > 0.55 else '#1a1a1a')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(['DST', 'SRC']); ax.set_yticklabels(['DST', 'SRC'] if ylab else [])
    ax.set_xlabel('code-progress span from', fontsize=9)
    if ylab: ax.set_ylabel('task span from')
    ax.tick_params(which='both', length=0)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks([0.5], minor=True); ax.set_yticks([0.5], minor=True)
    ax.grid(which='minor', color='white', linewidth=3, alpha=1); ax.grid(which='major', visible=False)
    ax.set_title(title, fontsize=9)

fig = plt.figure(figsize=(10.2, 3.1 * len(MODELS)))
for sf, m in zip(fig.subfigures(len(MODELS), 1, hspace=0.04), MODELS):
    T = tvp_summary(m); sf.suptitle(m, fontsize=11, fontweight='bold')
    axes = sf.subplots(1, 4, gridspec_kw=dict(wspace=0.35))
    draw(axes[0], tvp_grid(T, 'residual'), 'residual transfer', CMAP_RES, True)
    for c, (ax, v) in enumerate(zip(axes[1:], [1.0, 2.0, 4.0])):
        draw(ax, tvp_grid(T, 'kv', 'goal250', v), f'KV, 250 goal heads, V$\\times${v:g}', CMAP_KV, c == 0)
print('saved', F.save_fig(fig, GROUP, 'tvp_grids_by_lm')); plt.show()

In [ ]:
# ── KV transfer of the task span: one row per LM (by language direction | by datatype) ──
fig = plt.figure(figsize=(10, 3.3 * len(MODELS) + 0.4))
top, rows = fig.subfigures(2, 1, height_ratios=[0.4, 3.3 * len(MODELS)])
legend_top(top, 0.0)
for sf, m in zip(rows.subfigures(len(MODELS), 1, hspace=0.04), MODELS):
    sf.suptitle(m, fontsize=11, fontweight='bold')
    axes = sf.subplots(1, 2, sharey=True)
    sf.subplots_adjust(top=0.80, bottom=0.12)
    g = kv_grid(m)
    for ax, (axis, sub) in zip(axes, [('lang', 'by language direction'), ('dt', 'by datatype')]):
        cats, c, r = mean_over(g, axis)
        grouped_metric_bars(ax, cats, [LANG_LBL[x] for x in cats] if axis == 'lang' else cats, c, r)
        ax.set_title(sub, fontsize=10)
    print(m, 'overall causal %.3f readout %.3f' % tuple(np.mean([g[d][l][k] for d in DTS for l in LANGS]) for k in (0, 1)))
print('saved', F.save_fig(fig, GROUP, 'kv_barcharts_by_lm')); plt.show()

In [ ]:
# ── query transfer (gaze-select-restrict): one block per LM, rows = {order-matched, order-diff} ──
fig = plt.figure(figsize=(10, 5.6 * len(MODELS) + 0.4))
top, rows = fig.subfigures(2, 1, height_ratios=[0.4, 5.6 * len(MODELS)])
legend_top(top, 0.0)
for sf, m in zip(rows.subfigures(len(MODELS), 1, hspace=0.04), MODELS):
    sf.suptitle(m, fontsize=11, fontweight='bold')
    axes = sf.subplots(2, 2, sharey=True, gridspec_kw=dict(hspace=0.45))
    for i, v in enumerate(['order_matched', 'order_diff']):
        g = q_grid(m, v)
        for j, (axis, sub) in enumerate([('lang', 'by language direction'), ('dt', 'by datatype')]):
            cats, c, r = mean_over(g, axis)
            grouped_metric_bars(axes[i][j], cats, [LANG_LBL[x] for x in cats] if axis == 'lang' else cats, c, r)
            axes[i][j].set_title(f'{VAR_LBL[v]} · {sub}', fontsize=10)
        print(m, v, 'overall causal %.3f readout %.3f' % tuple(np.mean([g[d][l][k] for d in DTS for l in LANGS]) for k in (0, 1)))
print('saved', F.save_fig(fig, GROUP, 'q_barcharts_by_lm')); plt.show()